# System Realism: Dropouts, Stragglers, Bandwidth

In [1]:
import os
import sys
from pathlib import Path

root_dir = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import pandas as pd
from src.data import load_all_sites, load_pooled_data
from src.fl import run_federated
from src.system import run_system_sim

config = {
    "dataset": "fed_heart_disease",
    "seeds": list(range(42, 52)),
    "rounds": 50,
    "local_epochs": 1,
    "batch_size": 4,
    "lr": 0.001,
    "output_csv": str(root_dir / "results" / "e4.csv"),
}

Check: with p_drop = 0 and very large deadline, system simulation equals Stage 2 FedAvg exactly.

In [2]:
sites_data = load_all_sites()
pooled_te = load_pooled_data(train=False)

# Reference 5-round run
r_stage2 = run_federated(sites_data, pooled_te, seeds=[42], rounds=5, method="fedavg")
acc_stage2 = [r["metric_value"] for r in r_stage2 if r["site"] == "all"]

r_sys = run_system_sim(sites_data, pooled_te, seeds=[42], rounds=5, p_drop=0.0, deadline=1e6, scenario="ideal")
acc_sys = [r["metric_value"] for r in r_sys if r["site"] == "all"]

max_diff = max(abs(a - b) for a, b in zip(acc_stage2, acc_sys))
print(f"Parity check max difference: {max_diff:.12f}")
assert max_diff == 0.0, "Parity check failed!"
print("Check passed: Bit-for-bit equivalence verified.")

Parity check max difference: 0.000000000000
Check passed: Bit-for-bit equivalence verified.


Execute parameter sweeps: dropout probabilities, deadline thresholds with stragglers, and bandwidth heterogeneity.

In [3]:
records = []
seeds = config["seeds"]
rounds = config["rounds"]

scenarios = [
    {"scenario": "ideal", "p_drop": 0.0, "deadline": 1e6, "hetero_speed": False, "hetero_bw": False},
    {"scenario": "drop0.2", "p_drop": 0.2, "deadline": 1e6, "hetero_speed": False, "hetero_bw": False},
    {"scenario": "drop0.5", "p_drop": 0.5, "deadline": 1e6, "hetero_speed": False, "hetero_bw": False},
    {"scenario": "deadline_inf", "p_drop": 0.0, "deadline": 1e6, "hetero_speed": True, "hetero_bw": False},
    {"scenario": "deadline_2.0s", "p_drop": 0.0, "deadline": 2.0, "hetero_speed": True, "hetero_bw": False},
    {"scenario": "deadline_1.0s", "p_drop": 0.0, "deadline": 1.0, "hetero_speed": True, "hetero_bw": False},
    {"scenario": "hetero_bw_deadline_2.0s", "p_drop": 0.0, "deadline": 2.0, "hetero_speed": True, "hetero_bw": True},
]

for s in scenarios:
    print(f"Simulating scenario: {s['scenario']}...")
    recs = run_system_sim(
        sites_data, pooled_te, seeds=seeds, rounds=rounds,
        p_drop=s["p_drop"], deadline=s["deadline"],
        heterogeneous_speed=s["hetero_speed"],
        heterogeneous_bw=s["hetero_bw"],
        scenario=s["scenario"]
    )
    records.extend(recs)

df_e4 = pd.DataFrame(records)
os.makedirs(os.path.dirname(config["output_csv"]), exist_ok=True)
df_e4.to_csv(config["output_csv"], index=False)
print(f"Saved {len(df_e4)} rows to {config['output_csv']}")

Simulating scenario: ideal...


Simulating scenario: drop0.2...


Simulating scenario: drop0.5...


Simulating scenario: deadline_inf...


Simulating scenario: deadline_2.0s...


Simulating scenario: deadline_1.0s...


Simulating scenario: hetero_bw_deadline_2.0s...


Saved 17500 rows to C:\Users\Soumadipta Konar\Desktop\fl-hospital-consortium\results\e4.csv


Summary table showing accuracy, simulated wall-clock time, bytes transmitted, and client participation across system realism scenarios.

In [4]:
final_e4 = df_e4[(df_e4["round"] == config["rounds"]) & (df_e4["site"] == "all")].copy()
summary = final_e4.groupby("scenario").agg(
    acc_mean=("metric_value", "mean"),
    acc_std=("metric_value", "std"),
    sim_time=("sim_time_s", "mean"),
    bytes_sent=("bytes_sent", "mean"),
    avg_active=("n_clients_active", "mean")
).reset_index()

summary["accuracy"] = summary.apply(lambda r: f"{r['acc_mean']:.3f} +/- {r['acc_std']:.3f}", axis=1)
summary["sim_time_s"] = summary["sim_time"].round(1)
summary["bytes_sent"] = summary["bytes_sent"].astype(int)

cols = ["scenario", "accuracy", "sim_time_s", "bytes_sent", "avg_active"]
print(summary[cols].to_string(index=False))

               scenario        accuracy  sim_time_s  bytes_sent  avg_active
          deadline_1.0s 0.437 +/- 0.105        50.0       11200         0.0
          deadline_2.0s 0.623 +/- 0.078        84.7       16240         1.8
           deadline_inf 0.782 +/- 0.010       171.5       22400         4.0
                drop0.2 0.782 +/- 0.009    200150.1       17774         3.0
                drop0.5 0.781 +/- 0.012   3900128.1       11065         1.9
hetero_bw_deadline_2.0s 0.615 +/- 0.085        82.3       15960         1.7
                  ideal 0.782 +/- 0.010       155.5       22400         4.0
